# NDT7 (M-Lab) Data Prep — Myanmar Broadband + Mobile, Province x Quarter

Aggregates `../../../data/ndt7/mm/mlab_mm_clean.parquet` into province x quarter format, split
into Broadband and Mobile/Cellular parts, mirroring the same structure across all NDT7 "tigger"
countries (Cambodia/Thailand/Vietnam), ported here for Myanmar from its own standalone notebook
(`notebooks/ndt7/mm/mm_clean/ndt7_Myanmar_eda.ipynb`, 17/18 executed code cells with real
output — the underlying per-row schema and logic are verified/working, this notebook just
re-expresses the province x quarter roll-up in the tigger/DuckDB shape).

**DuckDB tile-binning** — same zoom-16 slippy-tile scheme as Ookla's own published tiles and
every other NDT7 "tigger" prep notebook, so `n_tiles`/`is_reliable` stay comparable across
Ookla and NDT7, and across countries: `total_tests >= 100 & n_tiles >= 5`.

**Province name mapping** — a small fix map (3 entries) is applied in §1.5 below: two spelling
variants (Sagaing/Tanintharyi) plus Naypyitaw folded into Mandalay (no separate reference row
exists for the capital territory, matching `notebooks/ookla/myanmar_eda.ipynb`'s own documented
convention). Found by actually running this notebook against the real raw parquet.

**Outputs:**
- `data/exports/ndt7_myanmar_province_quarterly.csv` — Broadband
- `data/exports/ndt7_mobile_myanmar_province_quarterly.csv` — Mobile/Cellular

**Not yet executed in this repo** — Myanmar raw data (`mlab_mm_clean.parquet`) is not
available locally; this notebook needs a run+verify pass before use (e.g. by whoever has the
full local dataset — the source notebook it's ported from was run there, with real output).


In [1]:
import duckdb
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

RAW_PARQUET = '../../../data/ndt7/mm/mlab_mm_clean.parquet'
MM_REF_CSV = '../../../data/reference/myanmar_reference.csv'

ZOOM = 16
N_TILES = 2 ** ZOOM
MIN_TILE_TESTS = 3


### 1. Tile-Binning + Province-Quarter Aggregation (DuckDB)

All heavy row-level work (filtering, quarter-labeling, zoom-16 mercator tile assignment, GROUP BY tile x quarter x type x network_type) happens in one DuckDB SQL query against the raw parquet — no Python-side batching.

In [2]:
sql = f"""
WITH filtered AS (
    SELECT
        mean_throughput_mbps,
        LEAST(min_rtt, 2000) AS min_rtt,
        latitude, longitude, type, network_type, province,
        date_part('year', date) AS yr,
        date_part('quarter', date) AS qtr
    FROM read_parquet('{RAW_PARQUET}')
    WHERE mean_throughput_mbps > 0
      AND latitude IS NOT NULL AND longitude IS NOT NULL
      AND province IS NOT NULL AND date IS NOT NULL
),
tiled AS (
    SELECT
        *,
        (CAST(yr AS VARCHAR) || '-Q' || CAST(qtr AS VARCHAR)) AS year_q,
        CAST(FLOOR((longitude + 180) / 360 * {N_TILES}) AS BIGINT) AS tile_x_raw,
        CAST(FLOOR((1 - (ln(tan(radians(LEAST(GREATEST(latitude, -85.05112878), 85.05112878))) + 1.0/cos(radians(LEAST(GREATEST(latitude, -85.05112878), 85.05112878)))) ) / pi()) / 2 * {N_TILES}) AS BIGINT) AS tile_y_raw
    FROM filtered
),
clipped AS (
    SELECT *,
        LEAST(GREATEST(tile_x_raw, 0), {N_TILES}-1) AS tile_x,
        LEAST(GREATEST(tile_y_raw, 0), {N_TILES}-1) AS tile_y
    FROM tiled
),
tile_id_cte AS (
    SELECT *, (CAST(tile_x AS VARCHAR) || '_' || CAST(tile_y AS VARCHAR)) AS tile_id
    FROM clipped
),
tile_agg AS (
    SELECT
        year_q, tile_id, type, network_type,
        AVG(mean_throughput_mbps) AS tile_mean,
        AVG(min_rtt) AS tile_lat,
        COUNT(*) AS test_count,
        mode(province) AS province
    FROM tile_id_cte
    GROUP BY year_q, tile_id, type, network_type
    HAVING COUNT(*) >= {MIN_TILE_TESTS}
)
SELECT * FROM tile_agg
"""

con = duckdb.connect()
tile_agg_all = con.execute(sql).df()
print(f"Tile x quarter x type x network rows (>= {MIN_TILE_TESTS} tests/tile): {len(tile_agg_all):,}")
print(f"Quarters covered: {sorted(tile_agg_all['year_q'].unique())}")
print(tile_agg_all['network_type'].value_counts())


Tile x quarter x type x network rows (>= 3 tests/tile): 707
Quarters covered: ['2023-Q1', '2023-Q2', '2023-Q3', '2023-Q4', '2024-Q1', '2024-Q2', '2024-Q3', '2024-Q4', '2025-Q1', '2025-Q2', '2025-Q3', '2025-Q4']
network_type
broadband    379
cellular     179
hosting      149
Name: count, dtype: int64


### 1.5 Province Name Fix — Raw → Reference

Verified against the real raw parquet (this section was originally assumed unnecessary before
the data was available locally -- it is needed). Two are spelling variants, one (Naypyitaw) is
a genuine reference-data gap consistent with an existing project decision (folded into Mandalay,
same as `notebooks/ookla/myanmar_eda.ipynb`).

In [3]:
# Real raw-data spelling/coverage fixes (found by actually running against the real parquet, not assumed):
# - Sagaing / Tanintharyi are alternate romanizations of the reference file's GADM-derived
#   'Saigang' / 'Tanitharyi' spellings.
# - Naypyitaw (the Union Territory) has no separate row in myanmar_reference.csv or the
#   geojson -- both were built from geoBoundaries ADM1, which does not carve Naypyidaw out
#   of Mandalay Region (see notebooks/ookla/myanmar_eda.ipynb's own documented decision to fold
#   it into Mandalay). Folding raw Naypyitaw test data into Mandalay here for consistency with
#   that established convention, rather than silently dropping the capital's data.
MM_PROVINCE_FIX = {
    'Sagaing': 'Saigang',
    'Tanintharyi': 'Tanitharyi',
    'Naypyitaw': 'Mandalay',
}
tile_agg_all['province'] = tile_agg_all['province'].replace(MM_PROVINCE_FIX)
print(f"Applied province fix map: {MM_PROVINCE_FIX}")


Applied province fix map: {'Sagaing': 'Saigang', 'Tanintharyi': 'Tanitharyi', 'Naypyitaw': 'Mandalay'}


### 2. Province Name Check — Raw → Reference (`province_en`)

After the §1.5 fix map, raw values should match `myanmar_reference.csv`'s `province_en`
directly. This cell validates that and drops any row that still doesn't match, same
defensive pattern as every other tigger country.

In [4]:
ref_check = pd.read_csv(MM_REF_CSV)
raw_provinces = set(tile_agg_all['province'].dropna().unique())
ref_provinces = set(ref_check['province_en'])
unmatched = raw_provinces - ref_provinces
if unmatched:
    print(f"WARNING — raw province values not found in reference, dropped: {unmatched}")
else:
    print("All raw province values matched to reference province_en directly (no remapping needed, "
          "same as Ookla's Myanmar pipeline and the source NDT7 notebook's own choropleth merge).")

print(f"Rows before province check: {len(tile_agg_all):,}")
tile_agg_all = tile_agg_all[tile_agg_all['province'].isin(ref_provinces)]
print(f"Rows after province check: {len(tile_agg_all):,}")


All raw province values matched to reference province_en directly (no remapping needed, same as Ookla's Myanmar pipeline and the source NDT7 notebook's own choropleth merge).
Rows before province check: 707
Rows after province check: 707


### 3. Province-Level Weighted Aggregation (per network type)

In [5]:
def build_province_quarterly(tile_agg_all, network_type, ref):
    tile_agg = tile_agg_all[tile_agg_all['network_type'] == network_type]
    print(f"[{network_type}] tile x quarter x type rows: {len(tile_agg):,}")

    dl = tile_agg[tile_agg['type'] == 'download']
    ul = tile_agg[tile_agg['type'] == 'upload']

    dl_stats = dl.groupby(['year_q', 'province']).apply(lambda g: pd.Series({
        'avg_d_mbps': np.average(g['tile_mean'], weights=g['test_count']),
        'avg_lat_ms_wt': np.average(g['tile_lat'], weights=g['test_count']),
        'total_tests': g['test_count'].sum(),
        'n_tiles': g['tile_id'].nunique(),
    }), include_groups=False).reset_index()

    ul_stats = ul.groupby(['year_q', 'province']).apply(lambda g: pd.Series({
        'avg_u_mbps': np.average(g['tile_mean'], weights=g['test_count']),
    }), include_groups=False).reset_index()

    master = pd.merge(dl_stats, ul_stats, on=['year_q', 'province'], how='outer')
    master = master.rename(columns={'year_q': 'quarter'})
    master['year'] = master['quarter'].str.slice(0, 4).astype(int)
    master['quarter.1'] = master['quarter'].str.slice(6, 7).astype(int)

    master['is_reliable'] = (master['total_tests'] >= 100) & (master['n_tiles'] >= 5)
    print(f"[{network_type}] province x quarter rows: {len(master)} | reliable: {master['is_reliable'].sum()} ({master['is_reliable'].mean():.1%})")

    master = master.merge(
        ref[['province_en', 'region', 'internet_tier', 'pop_2024', 'gdp_per_capita_raw_2021',
             'density_per_km2', 'gdp_per_capita_usd_ppp_2021', 'gdp_per_capita_thb_2021']],
        left_on='province', right_on='province_en', how='left'
    ).drop(columns=['province_en'])

    missing_ref = master[master['region'].isna()]['province'].unique()
    if len(missing_ref):
        print(f"[{network_type}] WARNING — provinces with no reference match: {list(missing_ref)}")

    return master


EXPORT_COLS = ['province', 'quarter', 'year', 'quarter.1', 'avg_d_mbps', 'avg_u_mbps',
               'avg_lat_ms_wt', 'total_tests', 'n_tiles', 'is_reliable', 'region',
               'internet_tier', 'pop_2024', 'gdp_per_capita_raw_2021', 'density_per_km2',
               'gdp_per_capita_usd_ppp_2021', 'gdp_per_capita_thb_2021']


In [6]:
ref = pd.read_csv(MM_REF_CSV)

---
## Part 1 — Broadband

In [7]:
broadband_master = build_province_quarterly(tile_agg_all, 'broadband', ref)
broadband_master.head()

[broadband] tile x quarter x type rows: 379
[broadband] province x quarter rows: 95 | reliable: 9 (9.5%)


,quarter,province,avg_d_mbps,avg_lat_ms_wt,total_tests,n_tiles,avg_u_mbps,year,quarter.1,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,2023-Q1,Bago,7.804803,235.738708,130.0,1.0,8.304277,2023,1,False,Lower Myanmar / Delta-Coastal,2,4867373,1242.72,124,5178.45,165710.4
1,2023-Q1,Chin,39.986269,85.245000,4.0,1.0,NaN,2023,1,False,Northern / Western Highlands,4,478801,1242.72,13,5178.45,165710.4
2,2023-Q1,Kachin,13.259478,133.999235,17.0,1.0,5.601712,2023,1,False,Northern / Western Highlands,4,1689441,1242.72,19,5178.45,165710.4
3,2023-Q1,Magway,8.187163,175.895857,14.0,1.0,10.071047,2023,1,False,Central Dry Zone,2,3917055,1242.72,87,5178.45,165710.4
4,2023-Q1,Mandalay,12.430088,186.900694,4761.0,6.0,8.412381,2023,1,True,Central Dry Zone,1,7325965,1242.72,163,5178.45,165710.4


In [8]:
out_bb = broadband_master[EXPORT_COLS].copy()
OUT_PATH_BB = '../../../data/exports/ndt7_myanmar_province_quarterly.csv'
out_bb.to_csv(OUT_PATH_BB, index=False)
print(f"Exported {len(out_bb)} rows -> {OUT_PATH_BB}")
out_bb.head(3)

Exported 95 rows -> ../../../data/exports/ndt7_myanmar_province_quarterly.csv


,province,quarter,year,quarter.1,avg_d_mbps,avg_u_mbps,avg_lat_ms_wt,total_tests,n_tiles,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,Bago,2023-Q1,2023,1,7.804803,8.304277,235.738708,130.0,1.0,False,Lower Myanmar / Delta-Coastal,2,4867373,1242.72,124,5178.45,165710.4
1,Chin,2023-Q1,2023,1,39.986269,NaN,85.245000,4.0,1.0,False,Northern / Western Highlands,4,478801,1242.72,13,5178.45,165710.4
2,Kachin,2023-Q1,2023,1,13.259478,5.601712,133.999235,17.0,1.0,False,Northern / Western Highlands,4,1689441,1242.72,19,5178.45,165710.4


---
## Part 2 — Mobile/Cellular

In [9]:
mobile_master = build_province_quarterly(tile_agg_all, 'cellular', ref)
mobile_master.head()

[cellular] tile x quarter x type rows: 179


[cellular] province x quarter rows: 53 | reliable: 1 (1.9%)


,quarter,province,avg_d_mbps,avg_lat_ms_wt,total_tests,n_tiles,avg_u_mbps,year,quarter.1,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,2023-Q1,Bago,12.548829,182.064087,160.0,2.0,4.633701,2023,1,False,Lower Myanmar / Delta-Coastal,2,4867373,1242.72,124,5178.45,165710.4
1,2023-Q1,Magway,0.557623,404.924667,6.0,1.0,0.172889,2023,1,False,Central Dry Zone,2,3917055,1242.72,87,5178.45,165710.4
2,2023-Q1,Mandalay,8.605738,266.552635,1004.0,4.0,4.028064,2023,1,False,Central Dry Zone,1,7325965,1242.72,163,5178.45,165710.4
3,2023-Q1,Saigang,15.757079,226.248118,17.0,1.0,4.405066,2023,1,False,Central Dry Zone,3,5325347,1242.72,57,5178.45,165710.4
4,2023-Q1,Shan,23.217634,129.402833,18.0,1.0,4.287337,2023,1,False,Eastern Highlands,3,5824432,1242.72,37,5178.45,165710.4


In [10]:
out_mb = mobile_master[EXPORT_COLS].copy()
OUT_PATH_MB = '../../../data/exports/ndt7_mobile_myanmar_province_quarterly.csv'
out_mb.to_csv(OUT_PATH_MB, index=False)
print(f"Exported {len(out_mb)} rows -> {OUT_PATH_MB}")
out_mb.head(3)

Exported 53 rows -> ../../../data/exports/ndt7_mobile_myanmar_province_quarterly.csv


,province,quarter,year,quarter.1,avg_d_mbps,avg_u_mbps,avg_lat_ms_wt,total_tests,n_tiles,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,Bago,2023-Q1,2023,1,12.548829,4.633701,182.064087,160.0,2.0,False,Lower Myanmar / Delta-Coastal,2,4867373,1242.72,124,5178.45,165710.4
1,Magway,2023-Q1,2023,1,0.557623,0.172889,404.924667,6.0,1.0,False,Central Dry Zone,2,3917055,1242.72,87,5178.45,165710.4
2,Mandalay,2023-Q1,2023,1,8.605738,4.028064,266.552635,1004.0,4.0,False,Central Dry Zone,1,7325965,1242.72,163,5178.45,165710.4


## Summary

- Input: Myanmar NDT7 raw test records, already province-joined + ISP-classified
- Output: province x quarter aggregates for Broadband and Mobile separately, tile-binned at
  Ookla's zoom-16 resolution, same `is_reliable` threshold as every Ookla country notebook and
  the other NDT7 "tigger" prep notebooks
- No province name remapping needed (raw values already match `myanmar_reference.csv`
  directly) — a defensive unmatched-value check runs anyway
- **Ported, not yet executed** — needs a run+verify pass against the full local dataset
